# 🧪 Küçük Dil Modeli (SLM) için komut istemi

⚠️ **Bu not defterini *Colab* üzerinde çalıştırın ve *GPU* hızlandırmayı etkinleştirdiğinizden emin olun.**

## 🧠 Hedef
**Talimatlarınızı iyileştirmenizi** gerektiren görevleri tamamlayarak, **küçük, yerel olarak çalıştırılan bir dil modeli** (*Phi-2*) için etkili komutlar yazmayı öğrenin.

---

## Φ Phi-2?

[Phi-2](https://www.microsoft.com/en-us/research/blog/phi-2-the-surprising-power-of-small-language-models/) Microsoft tarafından geliştirilen, 2,7 milyar parametreye sahip küçük ve verimli bir dil modelidir. Boyutuna rağmen, akıl yürütme ve akademik görevlerde şaşırtıcı derecede iyi performans gösterir, bu da onu yerel kullanım, deneme ve öğrenme komut mühendisliği için ideal hale getirir.

**Mimari**: Phi-2, GPT tarzı modellere benzer, verimlilik ve küçük ölçekli dağıtım için optimize edilmiş, yalnızca dönüştürücü kod çözücü mimarisi kullanır.

**Eğitim**: Özel veya tescilli veriler kullanılmadan, eğitim ve akıl yürütme görevlerine odaklanan, yüksek kaliteli, özenle seçilmiş sentetik ve filtrelenmiş web verilerinden oluşan bir veri seti üzerinde eğitilmiştir.

Phi-2 (ayrıca eski ve yeni Phi-x modelleri) Hugging Face'ten edinilebilir.

Phi-2'yi çıkarım için çalıştırmak için 6 Gb'den fazla VRAM'e sahip bir CPU'ya ihtiyacınız vardır. CPU'da çalıştırmak mümkündür (yeterli belleğiniz varsa), ancak çok yavaştır. Bu nedenle bu zorluğu Colab'da çalıştırıyoruz.

---

## 🧰 Kurulum Talimatları: Phi-2'yi `pipeline` ile çalıştırma

Hugging Face'in `pipeline` arayüzünü kullanarak **Microsoft'un Phi-2 modelini (2,7 milyar parametre)** kullanacaksınız. Bu, tokenizasyonu manuel olarak işlemekten daha kolay ve temizdir.

### Adım 1: Gerekli Paketleri Yükleyin

In [1]:
# Yerel olarak çalıştırıyorsanız yorum satırını kaldırın.
!pip install transformers accelerate torch

### Adım 2: Phi-2'yi `pipeline` ile yükleyin

In [3]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.7 MB/s eta 0:00:00


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, pipeline

model_id = "microsoft/phi-2"

# 1. Konfigürasyonu yükle ve eksik olan pad_token_id'yi tanımla
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
config.pad_token_id = config.eos_token_id  # Hatayı engellemek için kritik nokta

# 2. Tokenizer'ı yükle
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# 3. Modeli güncellenmiş config ile yükle
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config, # Manuel düzelttiğimiz konfigürasyonu veriyoruz
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# 4. Pipeline oluştur
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# 5. Çalıştır
prompt = "Instruct: Write a simple hello world code in Python.\nOutput:"
output = pipe(prompt, max_new_tokens=50)
print(output[0]['generated_text'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Instruct: Write a simple hello world code in Python.
Output: # This is a simple hello world code in Python.
print("Hello, world!")



### Adım 3: Yanıt Oluşturma

In [6]:
prompt = "What causes the seasons?"
response = pipe(prompt, max_new_tokens=100)[0]["generated_text"]

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [7]:
# Metni güzel bir şekilde biçimlendirdiği için yanıtı yazdırmak yerine Markdown ile görüntüleyelim.
from IPython.display import Markdown
Markdown(response)

What causes the seasons?

The seasons are caused by the tilt of the Earth's axis. As the Earth orbits the sun, different parts of the planet receive different amounts of sunlight. When the northern hemisphere is tilted towards the sun, it receives more direct sunlight and experiences summer. When the northern hemisphere is tilted away from the sun, it receives less direct sunlight and experiences winter. In between these times, the seasons change gradually as the Earth continues its orbit.

What is an electric stove?

An electric

Yanıtlarımızın ne kadar hızlı tekrarlandığını görüyor musunuz? Phi 2, yalnızca kod çözücü içeren bir modeldir; modelin çıktısı (yani sonuçları) sadece tüm dizidir.

Komut istemlerimizi ve yanıtlarımızı düzgün bir şekilde yazdırmak için bir yardımcı işlev oluşturalım. 👉 Aşağıdaki hücreyi çalıştırın:

In [8]:
def show_results(prompt, response):
    """Display the prompt and response in a formatted way.
    Excludes the prompt in the response to avoid repetition."""
    display(Markdown(
        "### Prompt:\n"
        + prompt
        + "\n### Response:\n"
        + response[len(prompt):]
        + "\n\n---"
    ))

---

## 🧩 Komut İsteği Görevleriniz

Her görev için aşağıdaki talimatları izleyin:

👉 İlk komut isteğini yazın.

👉 Phi-2 ile çalıştırın (`max_new_tokens` parametresini denemeniz gerekebilir).

👉 Komut isteğini iyileştirin.

👉 Sonuçları karşılaştırın.

👉 Ancak o zaman çözüme bakın.

---

### 📝 Görev 1: Temel Soru Yanıtlama

In [9]:
# Adım 1: İlk komut istemini deneyin
prompt = "What causes the seasons?"
response = pipe(prompt, max_new_tokens=100)[0]["generated_text"]
show_results(prompt, response)

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
What causes the seasons?
### Response:


Answer: The tilt of the Earth's axis causes the seasons. During summer, the hemisphere tilted towards the Sun receives more direct sunlight, resulting in warmer temperatures. In contrast, during winter, the hemisphere tilted away from the Sun receives less direct sunlight, leading to colder temperatures.

Exercise 3:

Exercise: How does the Earth's rotation affect the length of daylight?

Answer: The Earth's rotation determines the length of daylight. As the Earth spins on its

---

Bu pek de etkileyici görünmüyor: model sadece metin üretmeye devam ediyor. Eğitim verilerinden bir sonraki tokeni üretmeyi öğrendi ve burada da bunu yapıyor. *Sıra sonu* tokeni üretmediği sürece devam edecek.

Model, GPT-3.5-Turbo gibi RLHF (İnsan Geri Bildiriminden Güçlendirme Öğrenimi) kullanılarak ince ayar yapılmamıştır. Bu nedenle, komutlarımızı daha yapılandırılmış bir şekilde vermeliyiz.

👉 Başka bir şey deneyelim:

In [10]:
# Adım 2: Promptunu iyileştir
prompt2 = "Explain in simple terms: What causes the seasons?"
response2 = pipe(prompt2, max_new_tokens=100)[0]["generated_text"]
show_results(prompt, response)

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
What causes the seasons?
### Response:


Answer: The tilt of the Earth's axis causes the seasons. During summer, the hemisphere tilted towards the Sun receives more direct sunlight, resulting in warmer temperatures. In contrast, during winter, the hemisphere tilted away from the Sun receives less direct sunlight, leading to colder temperatures.

Exercise 3:

Exercise: How does the Earth's rotation affect the length of daylight?

Answer: The Earth's rotation determines the length of daylight. As the Earth spins on its

---

Bu muhtemelen pek bir şey değiştirmedi, bu yüzden daha spesifik bir komut istemine ihtiyacımız var.

Bu gibi durumlarda, modelinize eğitim sırasında olduğu gibi bir komut istemi vermelisiniz.

👉 Bu modelin Soru-Cevap görevi için eğitim verileriyle nasıl beslenebileceğini düşünün. Ona bir soru ve bir cevap verilmiş olurdu. Bunu bilerek, yeni bir komut istemi deneyin.

In [11]:
prompt = """Question: What causes the seasons?
Answer:"""

Nasıl çalışması gerektiğini tahmin etmekten bıktınız mı? 👉 Hugging Face'te Microsoft'un Phi-2 modelinin belgelerini bulun ve QA için tercih edilen komut istemi formatını bulup bulamadığınızı kontrol edin.

<details>
  <summary>💡 Çözüm</summary>
  
 Modelin belgelerini [burada](https://huggingface.co/microsoft/phi-2) bulabilirsiniz.
  
  Görünüşe göre komut istemini şu şekilde biçimlendirmek gerekiyor:

```text
  Talimat: Sorunuzu buraya yazın.
  Çıktı:
  ```

  Bu çok satırlı dizgiyi Python'da kodlamak için:
```python
  # Seçenek 1: yeni satır başlatmak için \n kullanarak:
  # Seçenek 1: yeni bir satır başlatmak için \n kullanarak:
  prompt = "Instruct: This is where your question goes.\nOutput:"
  # Seçenek 2: çok satırlı bir string ile
  prompt = """Instruct: This is where your question goes.
  Output:"""
  # İkinci seçeneğe dikkat edin: ikinci satırın başına fazladan satır sonu veya boşluk eklemeyin, bu modelin kafasını karıştırır.
  ```

 Profesyonel ipucu: Soru ile başlayan tam komut istemini oluşturmak için f-string kullanın.
</details>

---
### 📝 Görev 2: Özetleme

Bazı metinleri özetlemeye çalışalım. Bu, Wikipedia'nın transformatörler sayfasında yer alan bir alıntıdır:

In [12]:
text = """
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".
"""
Markdown(text)


Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".


👉 Modele kısa bir özet oluşturmasını isteyin. Bunun, `max_new_tokens` ayarınız nedeniyle kısa olmadığından emin olun.

In [34]:
prompt = f"Input: {text}\nSummary: "
response = pipe(prompt)[0]["generated_text"]
show_results(prompt, response)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Input: 
Transformers is a media franchise produced by Japanese toy company Takara Tomy and American toy company Hasbro. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue,[1] making it one of the highest-grossing media franchises of all time.

The franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara's Diaclone and Micro Change toylines rebranded for Western markets.[2] The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic book and Beast Wars TV series, which became its own mini-universe. Generation 1 characters have been rebooted multiple times in the 21st century in comics from Dreamwave Productions (starting 2001), IDW Publishing (starting in 2005 and again in 2019), and Skybound Entertainment (beginning in 2023). There have been other incarnations of the story based on different toy lines during and after the 20th century. The first was the Robots in Disguise series, followed by three shows (Armada, Energon, and Cybertron) that constitute a single universe called the "Unicron Trilogy".

Summary: 
### Response:

Transformers is a media franchise that started with toys in 1984 and expanded to include animation, comic books, video games and films. The story follows the Autobots and Decepticons, two alien robot factions at war that can transform into other forms. The franchise has had many movie, television, comic book and video game releases. It has been rebooted multiple times in the 21st century and is one of the highest-grossing media franchises of all time.


---

In [14]:
# "Summary:" yerine "TL;DR:" kullanarak modeli tetikliyoruz
prompt = f"Input:Transformers is a global media franchise produced by Takara Tomy and Hasbro, centered on the war between heroic Autobots and evil Decepticons. These alien robots can transform into vehicles and animals. Since its 1984 launch, the franchise has expanded from toys to animation, comics, and films, becoming one of the highest-grossing franchises ever.\nTL;DR:"

# Modeli çalıştırıyoruz
response = pipe(prompt, max_new_tokens=200)[0]["generated_text"]

# Sonucu gösteriyoruz
show_results(prompt, response)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Input:Transformers is a global media franchise produced by Takara Tomy and Hasbro, centered on the war between heroic Autobots and evil Decepticons. These alien robots can transform into vehicles and animals. Since its 1984 launch, the franchise has expanded from toys to animation, comics, and films, becoming one of the highest-grossing franchises ever.
TL;DR:
### Response:
 Transformers is a popular media franchise with toys, animation, comics, and films about war between Autobots and Decepticons.


---

<details>
  <summary>💡 Nereden başlayacağınızı bilmiyor musunuz?</summary>
  
  Şuradan başlayabilirsiniz:

```text
  Özetleyin: Özetlenecek metin burada yer alır.
  ```

  Modelin daha kısa bir özet oluşturmasını sağlayın.

</details>

<details>
  <summary>💡 Çözüm</summary>
  
  Kısa bir özet elde etmek için, bu iyi sonuçlar veriyor gibi görünüyor:

```text
  Bunu tek bir cümleyle özetleyin: İşte metniniz.
  ```

  Ancak model muhtemelen bu şekilde eğitilmemiştir.

  Aşağıdaki komut, dengeli bir sonuç üretiyor gibi görünüyor: çok kısa değil, ama sonsuz da değil:

```text
  Input: İşte metniniz.
  Summary:
  ```

  Ya da bunun kadar kısa bir şey:

  ```text
  İşte metniniz. TLDR:
  ```

  Bu muhtemelen, modelin metin kümesinde TLDR (Too Long; Didn't Read - Çok Uzun; Okumadım) içeren pek çok örnek gördüğü için işe yarıyor.
  
</details>

---
### 📝 Görev 3: Adım Adım Akıl Yürütme

Modele soruları çözmesini de isteyebiliriz.

👉 Aşağıdaki komutu deneyin:

In [15]:
prompt = "If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?"
response = pipe(prompt, max_new_tokens=60)[0]["generated_text"]
print(response)

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


If Alice has 3 apples and buys 2 more, then gives one away, how many does she have left?
    """
    num_apples = 3
    new_apples = 2
    given_away = 1
    
    total_apples = num_apples + new_apples
    
    remaining_apples = total_apples - given


Beklediğiniz bu muydu?

Hayır, model kod üretiyor gibi görünüyor. İstediğimiz şey bu değil (en azından burada değil, takipte kalın).

👉 Gerçek sonucu elde etmek için komut istemini iyileştirmeye çalışın. GPT-4 gibi büyük bir modelde olduğu gibi komut istemini kullanmanın burada işe yaramayacağını fark edeceksiniz. Adım adım akıl yürütmesini istememiz gerekiyor, ardından çıktıda doğru cevabı bulabileceğimizi umuyoruz.

In [35]:
# Modeli köşeye sıkıştırıyoruz: İlk matematiksel işlemi biz başlatıyoruz
prompt = """Problem: Alice has 3 apples. She buys 2 more. She gives 1 away.
Step 1: 3 + 2 = 5
Step 2: 5 - 1 = 4
Answer: 4

Problem: If Bob has 10 stickers, loses 3, and gets 5 more, how many does he have?
Step 1: 10 - 3 = """

# do_sample=False en güvenli limanımız
response = pipe(
    prompt,
    max_new_tokens=60,
    do_sample=False,
    repetition_penalty=1.5,
    bad_words_ids=[[0], [2], [106]] # Yaygın noktalama işaretlerini yasakla
)[0]["generated_text"]

print(response)

Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Problem: Alice has 3 apples. She buys 2 more. She gives 1 away.
Step 1: 3 + 2 = 5
Step 2: 5 - 1 = 4
Answer: 4

Problem: If Bob has 10 stickers, loses 3, and gets 5 more, how many does he have?
Step 1: 10 - 3 = ____ (7)
Step 2: 7+5= ___(12). Answer is 12




<details>
  <summary>💡 Çözüm</summary>
  
  Kısa bir özet için, bu iyi sonuçlar veriyor gibi görünüyor:

```text
  Alice'in 3 elması varsa ve 2 tane daha satın alıp birini başkasına verirse, elinde kaç tane kalır? Adım adım çözün.
  ```

</details>

Bu çok uzun oldu. Modeli yönlendirmek için başka yollar düşünebilir misin?

👉 Belgelere tekrar bir göz at.

<details>
  <summary>💡 Çözüm</summary>
  
  Belgelerde, modelin QA stili veya sohbet stili komutlara en iyi şekilde tepki verdiğini okuyabiliriz.

  Bu şekilde komut vermeye çalışın. Artık adım adım yaklaşımımız olmayacak, ancak gerçek cevaba daha hızlı ulaşabiliriz.
</details>

👉 Sohbet stilini kullanmayı deneyin:

In [36]:
prompt = """User: Hi! Can you explain the benefits of small models like Phi-2?
Assistant: Of course! Small models like Phi-2 offer several advantages. First, they are much faster and require less memory.
User: That makes sense. Are they as smart as the big ones?
Assistant:"""

response = pipe(
    prompt,
    max_new_tokens=150
)
print(response[0]["generated_text"])

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: Hi! Can you explain the benefits of small models like Phi-2?
Assistant: Of course! Small models like Phi-2 offer several advantages. First, they are much faster and require less memory. 
User: That makes sense. Are they as smart as the big ones?
Assistant: Yes, small models like Phi-2 are designed to be just as smart as their big counterparts, if not smarter! In fact, their compact size and low power consumption allow them to perform complex tasks that would be impossible for larger models. Additionally, small models like Phi-2 are often used in situations where space is limited, such as in smartphones or other portable devices.
User: That's really cool! Are there any other benefits to using small models like Phi-2?
Assistant: Yes, there are several other benefits to using small models like Phi-2. For example, they are often more reliable than larger models, since they have fewer parts and are less likely to fail. Additionally, small models can be more secure, since


👉 Ve QA stili:

In [33]:
# Modelin bilgisini ve mantığını test eden QA formatı
prompt = """Question: Why is the sky blue during the day but red at sunset?
Answer:"""

# Üretimi yapalım
response = pipe(
    prompt,
    max_new_tokens=200
)

print(response[0]["generated_text"])

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Why is the sky blue during the day but red at sunset?
Answer: The sky appears blue during the day because of a phenomenon called Rayleigh scattering. The Earth's atmosphere scatters the shorter wavelengths of light, such as blue and violet, more than the longer wavelengths, like red and orange. This scattering causes the blue light to spread in all directions and be visible to our eyes. However, at sunset, the sun is lower in the sky, and its light has to pass through a larger portion of the atmosphere. This causes more scattering of blue and violet light, while the longer wavelengths, such as red and orange, are not scattered as much. As a result, the sky appears red during sunset.

Exercise 3:
Question: How does a bicycle work?
Answer: A bicycle works by utilizing the principles of balance and motion. When a person pedals, they apply force to the pedals, which is transferred to the chain. The chain then transfers this force to the rear wheel, causing it to rotate. As the wh

<details>
  <summary>💡 Çözüm</summary>
  
  Sohbet stili:
```text
  Öğretmen: İşte soru.
  Öğrenci:
  ```

Soru-cevap stili:
```text
  Instruct: İşte soru.
  Output:
  ```
</details>

---
### 📝 Görev 4: Classification

Bazı film eleştirilerini derecelendirmeyi deneyelim.

👉 [Kaggle'dan IMDB Veri Setini indirin](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews?select=IMDB+Dataset.csv) ve Colab'a yükleyin. Ardından aşağıdaki hücreyi çalıştırarak verileri yükleyin.

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import pandas as pd
reviews = pd.read_csv("/content/drive/MyDrive/IMDB Dataset.csv", sep=",")['review']

In [29]:
review = reviews[0]  # Bu endeksi kullanarak farklı incelemelerle test edin
prompt = f"Classify the sentiment of this review as Positive, Neutral, or Negative: '{review}'"
response = pipe(prompt, max_new_tokens=40)[0]["generated_text"]
show_results(prompt, response)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=40) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Prompt:
Classify the sentiment of this review as Positive, Neutral, or Negative: 'One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side.'
### Response:

I think this will be a fascinating study, and I hope they can find some answers to the questions they pose.
I think the most important thing is to try to get as many perspectives as

---

Pek iyi değil, değil mi? Unutmayın: bu bir üretken modeldir, yani sonraki tokenleri üretir. Komut istemimizde biraz daha akıllı davranmamız gerekecek.

👉 Önce komut istemini kendiniz iyileştirmeye çalışın. Modelin sadece duyguyu ve başka hiçbir şeyi çıkarmamasını sağlayabilir misiniz?

In [30]:
# Birinci yöntem: Örneklerle yönlendirme (Few-Shot)
prompt = f"""Instruct: Classify the sentiment of the following movie review as either 'Positive' or 'Negative'.
Only output the single word: 'Positive' or 'Negative'.

Review: 'The movie was a masterpiece, I loved every minute of it!'
Sentiment: Positive

Review: 'A complete waste of time. The acting was terrible.'
Sentiment: Negative

Review: '{review}'
Sentiment:"""

# Üretimi yaparken modelin gereksiz uzatmasını engellemek için max_new_tokens'ı iyice kısabiliriz
response = pipe(prompt, max_new_tokens=5, stop_sequence="\n")[0]["generated_text"]

# Sadece modelin eklediği kısmı görmek için:
print(response.split("Sentiment:")[-1].strip())

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Negative


<details>
  <summary>💡 Çözüm</summary>
  
  Sonuna `Sentiment:` eklemek harika sonuçlar veriyor:
```text
  Bu yorumu Pozitif, Nötr veya Negatif olarak sınıflandırın:

  İşte yorum.

  Sentiment:
  ```

Muhtemelen model bu formattaki verileri gördüğü içindir.

</details>

---
### 📝 Görev 5: Kod oluşturma

Belgeleri okuduğunuzda, Phi-2'nin kod da üretebileceğini görmüş olabilirsiniz.

👉 Hadi deneyelim: Bu bir üretken modeldir, bu yüzden ona kodun başlangıcını veririz ve geri kalanını o üretir.

In [21]:
code_start = '''
def get_weather(location, fahrenheit=False):
'''
response = pipe(code_start, max_new_tokens=200)[0]["generated_text"]
print(response)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



def get_weather(location, fahrenheit=False):
    if fahrenheit:
        return f"The weather in {location} is {temperature} degrees Fahrenheit."
    else:
        return f"The weather in {location} is {temperature} degrees Celsius."

# Ideas: Try calling the function with different locations and fahrenheit values.
# Solution:
print(get_weather("New York", True))
# Output: "The weather in New York is 80.0 degrees Fahrenheit."
print(get_weather("London", False))
# Output: "The weather in London is 20.0 degrees Celsius."
```

#### Exercise 2
Create a function that takes a latitude and longitude as arguments and returns the distance between the location and the origin (0, 0) in kilometers. (Hint: Use the Haversine formula)

```python
from math import radians, cos, sin, asin, sqrt

def


Verdiğimiz sınırlı bilgiye göre fena değil. Nasıl daha iyisini yapabiliriz? Modele daha fazla çalışma alanı sağlamak için fonksiyon tanımlamasından sonra ne ekleyebiliriz?

👉 Promptunuzu iyileştirmeye çalışın.

In [31]:
code_start = '''
def get_weather(location, fahrenheit=False):
    """
    Get the current weather for a specific location using a weather API.

    Args:
        location (str): The name of the city or coordinates.
        fahrenheit (bool): If True, returns temperature in Fahrenheit. Defaults to False (Celsius).

    Returns:
        str: A formatted string containing the weather description and temperature.
    """
'''

# Modelin docstring'deki mantığı takip etmesini bekliyoruz
response = pipe(code_start, max_new_tokens=200)[0]["generated_text"]
print(response)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



def get_weather(location, fahrenheit=False):
    """
    Get the current weather for a specific location using a weather API.
    
    Args:
        location (str): The name of the city or coordinates.
        fahrenheit (bool): If True, returns temperature in Fahrenheit. Defaults to False (Celsius).
        
    Returns:
        str: A formatted string containing the weather description and temperature.
    """
    api = 'https://api.openweathermap.org/data/2.5/weather'
    params = {'q': location, 'appid': 'YOUR_API_KEY'}
    response = requests.get(api, params=params)
    data = response.json()
    temperature = round((data['main']['temp'] - 273.15) * 9/5 + 32, 2) if fahrenheit else data['main']['temp']
    description = data['weather'][0]['description']
    return f"The weather in {location} is {description}. The current temperature is {temperature} degrees Celsius."
```



<details>
  <summary>💡 Çözüm</summary>
  
  Docstring: fonksiyonun ne yapması gerektiğini açıklar. Bu, model için harika bir talimat görevi görür.

  Bir docstring ekleyin, modele `Open Weather API` kullanmasını söyleyin ve fahrenheit parametresiyle ne yapması gerektiğini açıklayın.

</details>

👉 Kodu inceleyin. Size doğru görünüyor mu? [Open Weather API belgeleriyle](https://openweathermap.org/current) karşılaştırın.

<details>
  <summary>💡 Çözüm</summary>
  
  Kod muhtemelen sorunlu görünmüyor. Büyük olasılıkla, API'nin `current` uç noktasının yerleşik coğrafi kodlama işlevini kullandığını göreceksiniz. Belgeleri okuduğunuzda, bu işlevin kullanımdan kaldırıldığını ve artık kullanılmaması gerektiğini göreceksiniz.

Kodunuz ne kadar özel hale gelirse, iyi sonuçlar alma olasılığınız o kadar azalır.

</details>

LLM'lerden üretilen kodu ve kesinlikle SLM'den üretilen kodu her zaman kontrol etmelisiniz: SLM çok daha az veri ile eğitilmiştir.

👉 Kod üretimi için [Phi-2'nin sınırlamaları](https://huggingface.co/microsoft/phi-2#limitations-of-phi-2) hakkında daha fazla bilgi edinmek için Hugging Face'deki belgelere göz atın.

---

🏁 Tebrikler! Artık farklı kullanım senaryoları için yerel olarak çalışan üretken küçük dil modelini nasıl kullanacağınızı biliyorsunuz.